# 05 - Block (AI Infra 视角)

本节从 **工程实现** 角度理解 Transformer Block：
- 残差连接与梯度流
- Pre-Norm vs Post-Norm
- 参数量与显存分析
- 深度扩展性

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. Block 核心 (30秒版)

```
Block = Attention + MLP + 残差连接

x ──────────────────┐
↓                   │
Norm → Attn ────────+ → x'
                    │
x' ─────────────────┐
↓                   │
Norm → MLP ─────────+ → 输出

GPT 模型 = N 个 Block 堆叠
```

## 2. 残差连接的工程意义

**核心作用**: 让梯度直通，支持深层网络

```
无残差:  梯度 = ∂L/∂x = ∂L/∂f × ∂f/∂x
         多层连乘 → 梯度消失/爆炸

有残差:  y = x + f(x)
         ∂y/∂x = 1 + ∂f/∂x
         至少有 1 保底!
```

In [2]:
# 梯度消失演示
n_layers = 100

# 无残差: 梯度连乘
grad_no_res = 1.0
for _ in range(n_layers):
    grad_no_res *= 0.95  # 每层梯度略小于 1

# 有残差: 梯度有直通路径
grad_with_res = 1.0  # 直通路径始终是 1

print(f"{n_layers} 层后的梯度:")
print(f"  无残差: {grad_no_res:.2e}  ← 消失了")
print(f"  有残差: {grad_with_res:.2e}  ← 保持")

100 层后的梯度:
  无残差: 5.92e-03  ← 消失了
  有残差: 1.00e+00  ← 保持


## 3. Pre-Norm vs Post-Norm (重要!)

```
Post-Norm (原版):           Pre-Norm (现代 LLM):

x → Attn → Add → Norm       x ─────────→ Add
      ↑                              ↑
      └──────x                Norm → Attn

问题: Norm 在主路径上          优点: 主路径无 Norm
      阻断梯度直通                   梯度直通!
```

### 为什么 Pre-Norm 训练更稳定?

1. 梯度直通: 残差路径上没有任何变换
2. 可以用更大的学习率
3. 几乎所有现代 LLM 都用 Pre-Norm

In [3]:
def norm(x):
    return F.rms_norm(x, (x.size(-1),))

# Post-Norm (原版 Transformer)
def post_norm_block(x, attn, mlp):
    x = norm(x + attn(x))   # Norm 在 Add 后面
    x = norm(x + mlp(x))
    return x

# Pre-Norm (现代 LLM)
def pre_norm_block(x, attn, mlp):
    x = x + attn(norm(x))   # Norm 在分支内
    x = x + mlp(norm(x))
    return x

print("Pre-Norm 的梯度路径:")
print("  x → Add → Add → Add → ...")
print("  主路径上只有加法，梯度直通!")

Pre-Norm 的梯度路径:
  x → Add → Add → Add → ...
  主路径上只有加法，梯度直通!


## 4. 参数量分析

```
每个 Block 的参数:
  Attention: 4 × dim² (Q, K, V, O)
  MLP:       8 × dim² (up, down)
  Norm:      0 (nanochat 无可学习参数)
  
  总计: 12 × dim²

N 层模型:
  Blocks:    N × 12 × dim²
  Embedding: vocab × dim
  LM Head:   dim × vocab (可与 Embedding 共享)
```

In [4]:
def estimate_model_params(n_layers, dim, vocab_size, tie_weights=True):
    """估算模型参数量"""
    block_params = 12 * dim * dim  # 每层
    total_blocks = n_layers * block_params
    
    embedding = vocab_size * dim
    lm_head = 0 if tie_weights else vocab_size * dim
    
    total = total_blocks + embedding + lm_head
    return total

# 常见模型配置
configs = {
    "GPT-2 Small":  (12, 768, 50257),
    "GPT-2 Medium": (24, 1024, 50257),
    "LLaMA 7B":     (32, 4096, 32000),
    "LLaMA 70B":    (80, 8192, 32000),
}

print("模型参数量估算:")
for name, (n_layers, dim, vocab) in configs.items():
    params = estimate_model_params(n_layers, dim, vocab)
    print(f"  {name:15s}: {params/1e9:.1f}B")

模型参数量估算:
  GPT-2 Small    : 0.1B
  GPT-2 Medium   : 0.4B
  LLaMA 7B       : 6.6B
  LLaMA 70B      : 64.7B


## 5. 显存分析

### 训练时总显存

```
总显存 = 模型参数 + 优化器状态 + 激活值 + 梯度
```

### 不用 Checkpoint 时

```
前向传播：所有层的中间激活值全部保存在显存中

激活值 (前向保存，反向消耗):
  每层保存:
    输入 x:             B × T × dim
    Attention 中间值:    B × H × T × T    (FlashAttention 可省掉)
    MLP 中间值:          B × T × 4×dim
  总计: N_layers 份全部同时驻留显存

参数梯度 (和参数同形状):
  Attention:  ~4 × dim²  × N_layers
  MLP:        ~8 × dim²  × N_layers

激活值梯度 (反向传播中流动的梯度信号):
  只需当前层的一份: B × T × dim (临时)

注意: 激活值 >> 参数梯度 >> 激活值梯度
```

### 用 Gradient Checkpoint 时 (现代 LLM 标配)

```
每层 Block 被 checkpoint 包裹:
  x = checkpoint(block, x)

前向: 只保存每层的【输入 x】, 层内中间值全部丢弃
反向: 逐层重算 forward，现算现用

保存的激活值:
  仅 N_layers 个输入:  N_layers × B × T × dim    ← 大幅减少!
  
不再保存:
  Attention 中间值:    B × H × T × T      ← 省掉
  MLP 中间值:          B × T × 4×dim      ← 省掉

代价: 反向时多算一次前向 (约 +30% 训练时间)
```

### 对比

```
                    不用 Checkpoint              用 Checkpoint
激活值显存      N_layers × (5×B×T×dim)      N_layers × (B×T×dim)
计算量          1× forward                   ~1.3× forward
```

In [ ]:
def training_memory_analysis(batch, seq_len, dim, n_heads, n_layers, dtype_bytes=2):
    """训练显存分析: 对比 Checkpoint vs 不用 Checkpoint"""
    
    # ========== 不用 Checkpoint ==========
    # 每层激活值
    attn_input = batch * seq_len * dim
    attn_matrix = batch * n_heads * seq_len * seq_len  # FlashAttn 可省
    mlp_hidden = batch * seq_len * 4 * dim
    
    per_layer_no_ckpt = (attn_input + attn_matrix + mlp_hidden) * dtype_bytes
    total_no_ckpt = per_layer_no_ckpt * n_layers
    
    # ========== 用 Checkpoint ==========
    # 每层只保存输入 x
    per_layer_ckpt = (batch * seq_len * dim) * dtype_bytes
    total_ckpt = per_layer_ckpt * n_layers
    
    # ========== 参数梯度 (两者相同) ==========
    param_grad = n_layers * 12 * dim * dim * dtype_bytes
    
    print(f"模型: {n_layers} 层, dim={dim}, batch={batch}, seq={seq_len}")
    print(f"{'='*55}")
    print(f"\n不用 Checkpoint:")
    print(f"  每层激活值:  {per_layer_no_ckpt/1e9:.2f} GB")
    print(f"  全部激活值:  {total_no_ckpt/1e9:.1f} GB")
    print(f"\n用 Checkpoint:")
    print(f"  每层激活值:  {per_layer_ckpt/1e9:.4f} GB (仅输入 x)")
    print(f"  全部激活值:  {total_ckpt/1e9:.1f} GB")
    print(f"\n参数梯度:      {param_grad/1e9:.1f} GB")
    print(f"\n节省: {(1 - total_ckpt/total_no_ckpt)*100:.0f}% 激活值显存")
    print(f"代价: 反向传播多算一次前向 (~+30% 训练时间)")

# LLaMA 7B
training_memory_analysis(
    batch=32, seq_len=4096,
    dim=4096, n_heads=32, n_layers=32
)

## 6. nanochat Block 实现

In [ ]:
# 简化版 (完整版见 nanochat/gpt.py)
class Block(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Linear(dim, dim, bias=False)  # 简化
        self.mlp_fc = nn.Linear(dim, 4 * dim, bias=False)
        self.mlp_proj = nn.Linear(4 * dim, dim, bias=False)
    
    def forward(self, x):
        # Pre-Norm + 残差
        x = x + self.attn(F.rms_norm(x, (x.size(-1),)))
        
        h = self.mlp_fc(F.rms_norm(x, (x.size(-1),)))
        h = F.relu(h).square()
        x = x + self.mlp_proj(h)
        
        return x

# 测试
block = Block(768)
x = torch.randn(2, 16, 768)
out = block(x)
print(f"输入: {x.shape} → 输出: {out.shape}")
print(f"形状不变，可以无限堆叠")

## 7. 面试常见问题

### Q1: 为什么用 Pre-Norm 而不是 Post-Norm?

**答**:
- Pre-Norm 的残差路径上没有任何变换
- 梯度可以直通，训练更稳定
- 可以使用更大的学习率
- 几乎所有现代 LLM 都用 Pre-Norm

---

### Q2: 残差连接的作用?

**答**:
- 解决梯度消失: y = x + f(x)，∂y/∂x 至少有 1
- 易于优化: 每层只学增量，而不是完整变换
- 深层网络: 使训练 100+ 层成为可能

---

### Q3: Block 中 Attention 和 MLP 的比例?

**答**:
- Attention: 4dim² (33%)
- MLP: 8dim² (67%)
- MLP 是参数大户！

---

### Q4: 为什么 Block 输出形状不变?

**答**:
- 允许任意层数堆叠
- 残差连接需要相同形状才能相加
- 所有变换都是 dim → dim

---

### Q5: 如何减少 Block 的显存占用?

**答**:
1. **Gradient Checkpointing**: 不存储中间激活，反向时重算
2. **FlashAttention**: 不存储 O(T²) 注意力矩阵
3. **混合精度**: 用 BF16/FP16 而不是 FP32
4. **Activation Offload**: 把激活值暂存到 CPU

## 8. 总结速查表

| 主题 | 要点 |
|------|------|
| **Pre-Norm** | Norm 在分支内，梯度直通 |
| **残差连接** | y = x + f(x)，梯度至少为 1 |
| **参数分布** | Attn 33% + MLP 67% = 12dim² |
| **形状不变** | 支持任意深度堆叠 |
| **显存瓶颈** | MLP 中间激活值 B×T×4×dim |

### nanochat Block 核心代码

```python
class Block(nn.Module):
    def __init__(self, config, layer_idx):
        self.attn = CausalSelfAttention(config, layer_idx)
        self.mlp = MLP(config)

    def forward(self, x, cos_sin, kv_cache):
        x = x + self.attn(norm(x), cos_sin, kv_cache)
        x = x + self.mlp(norm(x))
        return x
```